In [0]:
%py
print('heloo')

In [0]:
%sql
SHOW TABLES IN formula1_catalog.bronze

In [0]:
%run ../00-common/01.environment-config

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"


In [0]:

bronze_table

In [0]:
%sql
describe history formula1_catalog.bronze.results

In [0]:
# spark.read for aditonal options to read table data
#ciucuits_df=spark.read.option('versionAsOf',0).table(bronze_table)

In [0]:
results_df=spark.table(bronze_table)

In [0]:
results_df_selected=(results_df
                     .select("season",
                             "round",
                             "constructorId",
                             "driverId",
                             "date",
                             "raceName",
                             "grid",
                             "laps",
                             "number",
                             "points",
                             "position",
                             "positionText",
                             "status",
                             "ingestion_timestamp",
                             "source_file"
                             )
                     .drop("url"))

In [0]:
results_renamed_df=(
    results_df_selected
        .withColumnsRenamed ({
                     "driverId":"driver_id",
                     "date":"race_date",
                     "constructorId":"constructor_id",
                     "raceName":"race_name",
                     "grid":"grid_position",
                     "laps":"completed_laps",
                     "number":"car_number",
                     "points":"race_points",
                     "position":"funal_position",
                     "positionText":"final_position_text"
                     })  
                    
)

In [0]:
from pyspark.sql import functions as F
#circuits_renamed_nulldroped_df=circuits_renamed_df.filter("circuit_id is not  null")
#circuits_renamed_nulldroped_df=circuits_renamed_df.filter(circuits_renamed_df['circuit_id'].isNotNull())
results_valid_df=(results_renamed_df.
    filter(
    F.col('season').isNotNull()&
    F.col('round').isNotNull()&
    F.col('constructor_id').isNotNull() &
    F.col('driver_id').isNotNull() 
    )
)

In [0]:
display(results_valid_df.count())
display(results_renamed_df.count())
display("********")
display(results_renamed_df.count() - results_valid_df.count())


In [0]:
#circuits_distinct_df=circuits_renamed_nulldroped_df.distinct()
results_distinct_df=results_valid_df.dropDuplicates(["driver_id", "season", "round", "constructor_id"])
display(results_distinct_df)

In [0]:
display(results_valid_df.count()- results_distinct_df.count())

In [0]:

# duplicates = races_renamed_df[["season", "round"]].groupBy("season", "round").count().filter("count > 1")
# display(duplicates)

In [0]:
from pyspark.sql.functions import initcap
results_final_df=(results_distinct_df
    .withColumn('race_name',F.initcap(F.col('race_name')))
    
 )

In [0]:
display(results_final_df)

In [0]:
(
    results_final_df
        .write
        .format("delta")
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
select * from formula1_catalog.silver.results